# Missing Data Analysis: Default of Credit Card Clients

**Dataset**: Default of Credit Card Clients (UCI ML Repository #350)  
**Source**: Yeh, I. C., & Lien, C. H. (2009). *The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients in Taiwan.*  
**Size**: 30,000 clients × 25 features

This notebook demonstrates how `missingfcup` visualisations can help identify the underlying
mechanism of missing data — **MCAR**, **MAR**, or **MNAR** — by generating controlled synthetic
missingness on an otherwise complete dataset.

The dataset records six months of credit card payment history for Taiwanese clients. Its rich mix of
demographic (`AGE`, `SEX`, `EDUCATION`, `MARRIAGE`), financial (`LIMIT_BAL`, `BILL_AMT1–6`), and
behavioural (`PAY_0–6`, `PAY_AMT1–6`) features makes it ideal for crafting realistic missingness
scenarios where the ground truth is known and each mechanism's visual signature can be studied.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from missingfcup import MissingData
from mdatagen.multivariate.mMCAR import mMCAR
from mdatagen.multivariate.mMAR  import mMAR
from mdatagen.multivariate.mMNAR import mMNAR

default_credit = fetch_ucirepo(id=350)
X_raw = default_credit.data.features
y = default_credit.data.targets.iloc[:, 0].to_numpy()

# All features are already numeric — categorical codes (SEX, EDUCATION, MARRIAGE)
# are pre-encoded as integers in this dataset.
X = X_raw.copy().astype(float)
df = pd.concat([X, pd.Series(y, name="default")], axis=1)

print(f"Shape : {df.shape}")
print(f"Missing cells : {df.isna().sum().sum()}")
df.head()

## Complete Dataset — Baseline

Before injecting any missingness, we run all available `missingfcup` plots on the original
complete dataset. This establishes the **null baseline**: what every tool looks like when
there is nothing to detect. Plots that require at least one missing value are skipped.

In [ ]:
md_complete = MissingData(df)

md_complete.overall_missingness_barchart().show()
md_complete.missing_count_barchart().show()
md_complete.column_missing_rate_heatmap().show()
md_complete.heatmap().show()
md_complete.parallel_coordinates().show()
md_complete.scatterplot(x="AGE", y="LIMIT_BAL").show()

## Generating Synthetic Missingness

Since the dataset is complete, we inject missingness artificially using **`mdatagen`**, which
implements the three canonical mechanisms. We generate three separate copies of the data,
each with **30% missingness** injected via a different rule.

| Mechanism | Rule | Identifiability |
|-----------|------|-----------------|
| **MCAR** | Removed uniformly at random — independent of all values | Testable (Little's test) |
| **MAR**  | Missingness in some columns is driven by the *observed* values of other columns | Not directly testable; requires structural evidence |
| **MNAR** | Missingness depends on the *missing value itself* — extreme values are absent | Not identifiable from observed data alone |

Because we control the injection, the ground truth is known. The goal is to verify which
`missingfcup` tools correctly surface the expected diagnostic signal for each mechanism.

## 1. MCAR — Missing Completely At Random

Values are removed independently of all observed and unobserved data. No variable, row, or
value range has a higher probability of being missing than any other.

**Expected signatures:**
- Heatmap: uniform salt-and-pepper scatter — no row or column banding
- Missing-missing correlation: near-zero throughout — no pair of columns co-misses systematically
- Present-missing correlation: flat — observing one column carries no information about another's missingness
- Parallel coordinates: missing values (rendered as axis offsets) distributed uniformly across all value ranges
- Little's MCAR test: p > 0.05 (fail to reject)

In [ ]:
m_mcar = mMCAR(X=X, y=y, missing_rate=30)
data_m_mcar = m_mcar.random()
md_mcar = MissingData(data_m_mcar)

# --- Volume & distribution: how much is missing, and which columns? ---
md_mcar.overall_missingness_barchart().show()
md_mcar.missing_count_barchart().show()
md_mcar.column_missing_rate_heatmap().show()

# --- Structural pattern: is there a spatial structure to where values are missing? ---
md_mcar.heatmap().show()
md_mcar.barchart_intersection().show()
md_mcar.barchart_venn(selected_columns=["PAY_0", "BILL_AMT1", "PAY_AMT1"]).show()

# --- Correlation structure: do columns tend to be missing together? ---
md_mcar.missing_missing_heatmap().show()
md_mcar.present_missing_heatmap().show()
md_mcar.dendrogram().show()

# --- Value-level patterns: does the VALUE of a variable predict its own missingness? ---
md_mcar.parallel_coordinates().show()
md_mcar.scatterplot(x="AGE", y="LIMIT_BAL").show()

### What the Plots Reveal — MCAR

**Volume & Distribution**  
~30% of cells are missing, distributed roughly equally across all 25 columns. No single column
is disproportionately empty — the missing count barchart shows a flat, uniform profile. This
column-level uniformity is the first indicator of MCAR.

**Structural Pattern**  
The binary heatmap shows a classic salt-and-pepper texture: no horizontal bands of co-missing
rows, no vertical clusters of consistently empty columns. The barchart intersection confirms
this: many small, roughly equally-sized intersections with no dominant co-missing pattern.

**Correlation Structure**  
The missing-missing correlation heatmap is near-zero everywhere — no pair of columns
(e.g., `PAY_0` and `BILL_AMT1`) co-misses systematically. The dendrogram produces a shallow,
flat tree with arbitrary groupings and very short branch heights. The present-missing heatmap
is equally flat: observing any column carries no information about whether another will be
missing.

**Value-Level Patterns**  
In the parallel coordinates, the offset points (missing values rendered below each axis) are
spread uniformly across all value ranges of `AGE`, `LIMIT_BAL`, and the payment history
variables. There is no concentration at extremes. The scatterplot of `AGE` vs `LIMIT_BAL`
shows missing values scattered without any clustering in data space.

> **Mechanism Signature**: Absence of structure *is* the signal. Under MCAR, every tool
> designed to detect patterns produces a flat, uninformative output — which is precisely
> the correct result. This is the benchmark against which MAR and MNAR patterns stand out.

## 2. MAR — Missing At Random

Missingness depends on other *observed* variables but not on the value that is missing.
In this experiment, missingness in a subset of financial columns is conditioned on `AGE`:
older clients have a higher probability of missing `BILL_AMT` and `PAY` values — simulating
a scenario where older borrowers use a different reporting channel with incomplete data.

**Expected signatures:**
- Heatmap: structured row banding — rows with similar `AGE` values cluster as jointly incomplete
- Missing-missing correlation: high positive correlations between columns sharing the same MAR driver
- Present-missing correlation: strong asymmetric signal — `AGE` being observed predicts missingness in financial columns
- Parallel coordinates: missing values cluster in the high-`AGE` band when coloured by `AGE`
- Little's MCAR test: p < 0.05 (reject MCAR)

In [ ]:
m_mar = mMAR(X=X, y=y, n_xmiss=5)
data_m_mar = m_mar.random(missing_rate=30)
md_mar = MissingData(data_m_mar)

# --- Volume & distribution ---
md_mar.overall_missingness_barchart().show()
md_mar.missing_count_barchart().show()
md_mar.column_missing_rate_heatmap().show()

# --- Structural pattern ---
md_mar.heatmap().show()
md_mar.barchart_intersection().show()
md_mar.barchart_venn(selected_columns=["PAY_0", "BILL_AMT1", "PAY_AMT1"]).show()

# --- Correlation structure ---
md_mar.missing_missing_heatmap().show()
md_mar.present_missing_heatmap().show()
md_mar.dendrogram().show()

# --- Value-level patterns: colour by the MAR driver (AGE) to reveal the conditioning ---
md_mar.parallel_coordinates(missingness_color_column="AGE").show()
md_mar.scatterplot(x="AGE", y="LIMIT_BAL").show()

### What the Plots Reveal — MAR

**Volume & Distribution**  
Total missingness remains ~30%, but the column-level distribution is no longer uniform.
The conditioned columns (`BILL_AMT`, `PAY`) carry elevated missing rates while the driver
column (`AGE`) remains fully observed — a structural asymmetry already visible in the
missing count barchart.

**Structural Pattern**  
The heatmap shows a striking change: **horizontal bands** of co-missing rows appear. Clients
with similar `AGE` values are simultaneously incomplete across multiple financial columns.
The barchart intersection shifts toward a small number of large, dominant co-missing patterns —
very different from the dispersed MCAR profile.

**Correlation Structure**  
The missing-missing correlation heatmap reveals strong positive correlations between
`BILL_AMT1–6` and `PAY_AMT1–6`: these columns miss together because their missingness is
driven by the same observed variable (`AGE`). The dendrogram produces tight, meaningful
clusters of financially-related columns.

Critically, the **present-missing heatmap** reveals the MAR diagnostic: `AGE` being present
in a row is strongly associated with missingness in `LIMIT_BAL` and financial behaviour
columns. This asymmetric cross-signal — an observed column predicting another's missingness —
is the direct evidence of MAR.

**Value-Level Patterns**  
The parallel coordinates, coloured by `AGE` missingness, shows the missing-value offsets
clustering in the high-`AGE` band. The scatterplot of `AGE` vs `LIMIT_BAL` shows missing
`LIMIT_BAL` concentrated among older clients: a horizontal strip of offset points on the
right side of the `AGE` axis.

> **Mechanism Signature**: MAR produces structured row clusters in the heatmap, high
> inter-column missingness correlations, and a strong asymmetric present-missing signal.
> Identifying the driver column in the present-missing heatmap is the key diagnostic step.

## 3. MNAR — Missing Not At Random

Missingness depends on the *missing value itself*. Here, extreme values of financial columns
are more likely to be absent — simulating a scenario where clients with very high bill amounts
or extreme repayment patterns are less likely to have those values recorded (e.g., opt-out
reporting, data suppression for outliers).

This mechanism is fundamentally unidentifiable from observed data alone: the information needed
to explain the missingness is precisely the information that is gone.

**Expected signatures:**
- Heatmap: superficially similar to MCAR — no structured banding from observed predictors
- Missing-missing correlation: moderate if columns share similar threshold rules; weaker than MAR
- Present-missing correlation: weak — no observed column cleanly predicts the missingness
- Parallel coordinates: missing values concentrate at extreme ends of the value range
- Little's MCAR test: p < 0.05 (rejects MCAR, but cannot distinguish from MAR)

In [ ]:
m_mnar = mMNAR(X=X, y=y, threshold=1, n_xmiss=8)
data_m_mnar = m_mnar.random(missing_rate=30)
md_mnar = MissingData(data_m_mnar)

# --- Volume & distribution ---
md_mnar.overall_missingness_barchart().show()
md_mnar.missing_count_barchart().show()
md_mnar.column_missing_rate_heatmap().show()

# --- Structural pattern ---
md_mnar.heatmap().show()
md_mnar.barchart_intersection().show()
md_mnar.barchart_venn(selected_columns=["PAY_0", "BILL_AMT1", "PAY_AMT1"]).show()

# --- Correlation structure ---
md_mnar.missing_missing_heatmap().show()
md_mnar.present_missing_heatmap().show()
md_mnar.dendrogram().show()

# --- Value-level patterns: the clearest MNAR signal lives here ---
md_mnar.parallel_coordinates().show()
md_mnar.scatterplot(x="AGE", y="LIMIT_BAL").show()

### What the Plots Reveal — MNAR

**Volume & Distribution**  
Missing rates are elevated in the affected columns but the overall volume is similar to the
other scenarios. Unlike MAR, the column with extreme values (e.g., `BILL_AMT1`) shows a higher
missing rate than columns with more uniform distributions.

**Structural Pattern**  
The heatmap is the key difference from MAR: there is **no structured row banding**. Missing
values appear scattered without obvious demographic grouping — because the mechanism is driven
by each column's own values, not by a shared observed predictor. At first glance this resembles
MCAR, and this visual ambiguity is the central challenge of MNAR.

**Correlation Structure**  
The missing-missing heatmap shows moderate correlations: columns affected by similar
threshold rules (high-bill-amount columns) tend to miss together. However the pattern is
less clean and structured than the MAR case. The present-missing heatmap shows a **weaker
signal** than MAR — no single observed column cleanly predicts missingness in others,
because the true predictor (the missing column's own extreme value) is unobserved.

**Value-Level Patterns**  
This is where MNAR leaves its clearest trace. In the parallel coordinates, the offset points
for `BILL_AMT` columns are not spread uniformly — they cluster toward the **high end** of the
visible range. Clients who *do* have high bill amounts are less likely to have those values
recorded. The scatterplot of `AGE` vs `LIMIT_BAL` shows missing `LIMIT_BAL` values slightly
concentrated among clients with higher credit limits — a subtle but real MNAR signature.

> **Mechanism Signature**: MNAR partially resists visual detection from correlation-based
> tools. Its most accessible signature — value-level clustering in parallel coordinates and
> scatterplots — requires knowing which columns to examine. This is why domain knowledge is
> essential when MNAR is suspected.

## Comparative Analysis — Little's MCAR Test

Little's test evaluates whether the missingness pattern is consistent with MCAR by comparing
the observed pattern against what would be expected under complete randomness.

| Result | Interpretation |
|--------|-----------------|
| **p > 0.05** | Fail to reject MCAR — data *may* be MCAR |
| **p ≤ 0.05** | Reject MCAR — consistent with MAR or MNAR |

**Important**: Little's test only distinguishes MCAR from non-MCAR. It **cannot** distinguish
MAR from MNAR. The visual tools are essential for going further.

In [ ]:
results = {
    "MCAR": md_mcar.littles_mcar_test(),
    "MAR":  md_mar.littles_mcar_test(),
    "MNAR": md_mnar.littles_mcar_test(),
}

pd.DataFrame({k: v[["chi2", "df", "p_value"]] for k, v in results.items()}).T

## Conclusions

### Plot Utility by Mechanism

| Plot | MCAR | MAR | MNAR |
|------|------|-----|------|
| Binary heatmap | Baseline (random scatter) | ★★★ Clear row banding | ★ Looks like MCAR |
| Missing-missing correlation | ★ Near-zero | ★★★ Strong, interpretable clusters | ★★ Moderate, noisy |
| Present-missing heatmap | ★ Flat | ★★★ Driver column stands out | ★ Weak signal |
| Parallel coordinates | ★ Uniform offsets | ★★ Banded by driver variable | ★★★ Extremal value clustering |
| Scatterplot | ★ Random | ★★ Spatial clustering by driver | ★★ Value-range bias |
| Little's MCAR test | ★★★ Confirmed (p > 0.05) | ★★★ Rejected (p < 0.05) | ★★ Rejected — indistinguishable from MAR |

### Key Takeaways

1. **MCAR is the null case.** Absence of visual structure is the correct finding. All tools
   produce flat, uninformative output — and that flatness is itself informative.

2. **MAR is visually detectable.** The present-missing heatmap is the most powerful single
   tool: it directly reveals which observed column drives the missingness of others. Row
   banding in the binary heatmap and tight clusters in the missing-missing correlation
   heatmap provide corroborating evidence.

3. **MNAR resists correlation-based tools.** Its signature lives in the value distribution
   of the missing data, best accessed through parallel coordinates and targeted scatterplots.
   Domain knowledge about which variables are susceptible to self-selective missingness is
   essential for correct diagnosis.

4. **Little's test confirms but does not diagnose.** It correctly distinguishes MCAR from
   non-MCAR, but the visual tools are required to determine *which* non-random mechanism
   is operating.

5. **No single plot is sufficient.** The combination of structural (heatmap), correlation-based
   (missing-missing, present-missing), and value-level (parallel coordinates, scatterplot)
   tools provides the most complete and reliable diagnostic picture.